In [ ]:
## first we will read in necessary python libraries and intialize the GEE module
## you will have to authenticate GEE on your machine to use this code

import ee
import geopandas as gpd
from shapely.geometry import box
import time

## if you need to authenticate GEE use this code
# ee.Authenticate()
ee.Initialize(project="ee-tlasharr")

## time period -- match your spline raster years
start_year = 2003
end_year   = 2025



Successfully saved authorization token.


In [ ]:
## study area bounding box -- north-south transect MT to AZ
## matches the updated md_poly in 00_Config.R
aoi = ee.Geometry.BBox(-112.5, 31.3, -109.5, 49.0)

## get daymet v4 -- prcp (mm/day), swe (kg/m2), tmin (C), tmax (C)
## we need prcp to identify rainfall events, swe for snowmelt events,
## and tmin to determine rain-vs-snow and days above freezing
daymet = (ee.ImageCollection("NASA/ORNL/DAYMET_V4")
          .filterBounds(aoi)
          .select(['prcp', 'swe', 'tmin', 'tmax']))

## export one image per year as a multiband tif -- one band per day
## this is much more efficient than one export per day (avoids the
## thousands of tasks your current code generates)
def export_year(year):
    start = f"{year}-01-01"
    end   = f"{year}-12-31"

    ## filter to year, clip to aoi
    yr_coll = daymet.filterDate(start, end).map(
        lambda img: img.clip(aoi)
    )

    ## stack all daily images into a single multiband image
    ## each band is named prcp_YYYY-MM-DD etc via system:index
    def add_date_to_bandname(img):
        date_str = ee.Date(img.get('system:time_start')).format('YYYY_MM_dd')
        bands    = img.bandNames()
        new_names = bands.map(lambda b: ee.String(b).cat('_').cat(date_str))
        return img.rename(new_names)

    yr_stack = yr_coll.map(add_date_to_bandname).toBands()

    task = ee.batch.Export.image.toDrive(
        image       = yr_stack,
        description = f'DAYMET_DroughtSurfing_{year}',
        folder      = 'DAYMET_DroughtSurfing',
        fileNamePrefix = f'DAYMET_{year}',
        region      = aoi,
        scale       = 1000,       ## daymet native resolution is ~1km
        crs         = 'EPSG:4326',
        fileFormat  = 'GeoTIFF',
        maxPixels   = 1e13
    )
    task.start()
    print(f"export started: DAYMET_{year}")

for yr in range(start_year, end_year + 1):
    export_year(yr)
    time.sleep(1)   ## small pause to avoid rate limiting

print("all export tasks submitted")

export started: DAYMET_2003
export started: DAYMET_2004
export started: DAYMET_2005
export started: DAYMET_2006
export started: DAYMET_2007
export started: DAYMET_2008
export started: DAYMET_2009
export started: DAYMET_2010
